**Generate Dataset**
Automation codes to run localhost and download data from stirsimul

**Imports and Model Setup**

In [ ]:
import torch
print("PyTorch version:", torch.__version__)  # Should show CUDA version
print("CUDA version in PyTorch:", torch.version.cuda)  # Should match installed CUDA (e.g., 12.1)
print("CUDA available:", torch.cuda.is_available())  # Should return True
print("Number of GPUs:", torch.cuda.device_count())  # Should be > 0
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import tifffile
from PIL import Image
import glob
from pathlib import Path
import timm.layers
import tqdm
from patchify import patchify 

torch.cuda.is_available()

In [ ]:
from mobile_sam import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

model_type = "vit_t"
sam_checkpoint = "weights/mobile_sam.pt"
device = "cpu" # set device to cpu temporarily for dataset transforms

mobile_sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
mobile_sam.to(device=device)

predictor = SamPredictor(mobile_sam)
mask_generator = SamAutomaticMaskGenerator(mobile_sam)

**Generate Dataset**

In [ ]:
# Load and stack color images into TIFF format
rawSet = [np.array(Image.open(file)) for file in sorted(glob.glob("captures/raw/raw_*.png"))]
maskedSet = [np.array(Image.open(file)) for file in sorted(glob.glob("captures/masked/masked_*.png"))]

# Save as multi-page TIFF stacks with RGB for color images
tifffile.imwrite("captures/raw_stack.tif", np.array(rawSet), photometric='rgb')
tifffile.imwrite("captures/masked_stack.tif", np.array(maskedSet), photometric='rgb')

In [ ]:
# SHAPE : (N, H, W, C)
large_rawSet_temp = tifffile.imread("captures/raw_stack.tif")
large_maskedSet_temp = tifffile.imread("captures/masked_stack.tif")

print(large_rawSet_temp.shape) 
print(large_maskedSet_temp.shape)

In [ ]:
large_maskedSet = large_maskedSet_temp[..., 0]  # Take only the RGB channel
large_maskedSet = large_maskedSet.astype(np.uint8) 

large_rawSet = large_rawSet_temp[:, :, :, :3]
large_maskedSet = large_maskedSet[:, :, :]
print(large_rawSet.shape) 
print(large_maskedSet.shape)

In [ ]:
import cv2

def applyMorphology(data, radius_erosion, iter_erosion, radius_dilation, iter_dilation, operation):

    """
    Apply morphology to binary image using opencv library

    Parameters:
        data: masked binary data in numpy array format [H*W]
        radius_erosion: int value of erosion radius
        iter_erosion: int value of repetition for erosion
        radius_dilation: int value of dilation radius
        iter_dilation: int value of repetition for dilation
        operation: if set as "opening", erosion-dilation. if set as "closing",  dilation-erosion

    Returns:
        numpy array[H*W]
    """

    kernel_erosion = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (radius_erosion, radius_erosion))
    kernel_dilation = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (radius_dilation, radius_dilation))

    if operation == "opening":
        temp = cv2.erode(data, kernel_erosion, iterations = iter_erosion)
        output = cv2.dilate(temp, kernel_dilation, iterations = iter_dilation)
    elif operation == "closing":
        temp = cv2.dilate(data, kernel_dilation, iterations = iter_dilation)
        output = cv2.erode(temp, kernel_erosion, iterations = iter_erosion)
    else: print("invalid operation. operation should be 'opening' or 'closing'")

    return output

id = 102
smooth_data = applyMorphology(large_maskedSet[id], 1, 1, 3, 2, "opening")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(large_rawSet[id]) 
axes[0].set_title("Raw Image")

axes[1].imshow(large_maskedSet[id], cmap = "gray")
axes[1].set_title("Masked Image")

axes[2].imshow(smooth_data, cmap = "gray") 
axes[2].set_title("Smoothed Image")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.show()

In [ ]:
smooth_maskedSet = []

for data in large_maskedSet:
    smooth_data = applyMorphology(data, 1, 1, 3, 2, "opening")
    smooth_maskedSet.append(smooth_data)

In [ ]:
#Desired patch size for smaller images and step size.
patch_size = 256
step = 256

all_raw_patches = []
for raw in range(large_rawSet.shape[0]):
    large_raw = large_rawSet[raw]
    
    # Patchify with (patch_size, patch_size, 4) to include RGBA channels
    patches_raw = patchify(large_raw, (patch_size, patch_size, 3), step=step)

    # Iterate through the patches and append to all_img_patches
    for i in range(patches_raw.shape[0]):
        for j in range(patches_raw.shape[1]):
            single_patch_raw = patches_raw[i, j, 0, :, :, :]  # Extract each patch with all channels
            all_raw_patches.append(single_patch_raw)

# Convert list of patches to numpy array
rawSet = np.array(all_raw_patches)

# Process masks, assuming they are single-channel (binary)
all_masked_patches = []
for large_masked in smooth_maskedSet:
    # If the mask has an extra channel dimension, reduce to single channel
    if large_masked.ndim == 3 and large_masked.shape[-1] == 4:
        large_masked = large_masked[:, :, 0]  # Take only one channel

    # Patchify for 2D patches on single-channel masks
    patches_masked = patchify(large_masked, (patch_size, patch_size), step=step)

    for i in range(patches_masked.shape[0]):
        for j in range(patches_masked.shape[1]):
            single_patch_masked = patches_masked[i, j, :, :]  # Extract as 2D array
            single_patch_masked = (single_patch_masked / 255).astype(np.uint8)  # Normalize to binary if needed
            all_masked_patches.append(single_patch_masked)

# Convert list of patches to numpy array
maskedSet = np.array(all_masked_patches)

# Create a list to store the indices of non-empty masks
valid_indices = [i for i, mask in enumerate(maskedSet) if mask.max() != 0]

# Filter the image and mask arrays to keep only the non-empty pairs
filtered_raw = rawSet[valid_indices]
filtered_masked = maskedSet[valid_indices]
print("Raw shape:", filtered_raw.shape)  # e.g., (num_frames, height, width, num_channels)
print("Masked shape:", filtered_masked.shape)


In [ ]:
import torch

#Get bounding boxes from mask.
def get_bounding_box(ground_truth_map):
  # get bounding box from mask
  y_indices, x_indices = np.where(ground_truth_map > 0)
  x_min, x_max = np.min(x_indices), np.max(x_indices)
  y_min, y_max = np.min(y_indices), np.max(y_indices)
  # add perturbation to bounding box coordinates
  H, W = ground_truth_map.shape
  x_min = max(0, x_min - np.random.randint(0, 20))
  x_max = min(W, x_max + np.random.randint(0, 20))
  y_min = max(0, y_min - np.random.randint(0, 20))
  y_max = min(H, y_max + np.random.randint(0, 20))
  #bbox = np.array([x_min, y_min, x_max, y_max])
  bbox = np.array([0, 0, 255, 255])


  return bbox

# Add bounding boxes to the dataset
filtered_boxes = []
for mask in filtered_masked:
  filtered_boxes.append(get_bounding_box(mask))


In [ ]:
from datasets import Dataset
from PIL import Image

# Convert the NumPy arrays to Pillow images and store them in a dictionary
dataset_dict = {
    "image": [Image.fromarray(raw) for raw in filtered_raw],
    "label": [Image.fromarray(mask) for mask in filtered_masked],
}

# Create the dataset using the datasets.Dataset class
dataset = Dataset.from_dict(dataset_dict)
print(dataset.shape)

In [ ]:
raw_num = random.randint(0, filtered_raw.shape[0]-1)
example_raw = dataset[raw_num]["image"]
example_masked = dataset[raw_num]["label"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot the first image on the left
axes[0].imshow(np.array(example_raw))  # Assuming the first image is grayscale
axes[0].set_title("Image")

# Plot the second image on the right
axes[1].imshow(example_masked, cmap='gray')  # Assuming the second image is grayscale
axes[1].set_title("Mask")

# Hide axis ticks and labels
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

# Display the images side by side
plt.show()

**Dataset Processor for MobileSAM**

In [ ]:
import torch
from mobile_sam.utils import transforms

# Image transforms
transform = transforms.ResizeLongestSide(mobile_sam.image_encoder.img_size)

transformed_images = []
for raw in filtered_raw:
    transformed_image = torch.as_tensor(transform.apply_image(raw), device="cpu")
    transformed_image = transformed_image.permute(2, 0, 1)
    transformed_images.append(transformed_image)

# Box transforms
transformed_boxes = []
for box in filtered_boxes:
    transformed_box = torch.as_tensor(transform.apply_boxes(box, (256, 256)), dtype=torch.float, device="cuda")[None, :]
    transformed_boxes.append(transformed_box)

# GT Mask transforms
transformed_masks = []
for mask in filtered_masked:
    transformed_mask = torch.from_numpy(mask).float()
    transformed_masks.append(transformed_mask)

In [ ]:
# Splitting data set(train:val:test = 8:1:1)

from sklearn.model_selection import train_test_split

train_images, temp_images, train_boxes, temp_boxes, train_masks, temp_masks = train_test_split(transformed_images, transformed_boxes, transformed_masks, test_size=0.2, random_state=42)
val_images, test_images, val_boxes, test_boxes, val_masks, test_masks = train_test_split(temp_images, temp_boxes, temp_masks, test_size=0.5, random_state=42)

In [ ]:
batch_size = 8
train_batch = []
val_batch = []
current_batch = []

#train_batch fill
for img, boxes, mask in zip(train_images, train_boxes, train_masks):
    sample = {
        "image": img,  # The image tensor
        "original_size": (256, 256),  # Original size of the image
        "boxes": boxes.requires_grad_().to("cuda"),  # Boxes for this image
        "ground_truth_mask": mask.to("cuda").requires_grad_()
    }
    current_batch.append(sample)

    if len(current_batch) == batch_size:
        train_batch.append(current_batch)
        current_batch = []

if current_batch:
    train_batch.append(current_batch)

#val_batch fill
for img, boxes, mask in zip(val_images, val_boxes, val_masks):
    sample = {
        "image": img,  # The image tensor
        "original_size": (256, 256),  # Original size of the image
        "boxes": boxes.requires_grad_().to("cuda"),  # Boxes for this image
        "ground_truth_mask": mask.to("cuda").requires_grad_()
    }
    current_batch.append(sample)

    if len(current_batch) == batch_size:
        val_batch.append(current_batch)
        current_batch = []

if current_batch:
    val_batch.append(current_batch)

In [ ]:
# Save dataset
torch.save(train_batch, './captures/train_batch.pt')
torch.save(val_batch, './captures/val_batch.pt')


**Reset Model for Training**

In [ ]:
train_batch = torch.load('./captures/train_batch.pt')
val_batch = torch.load('./captures/val_batch.pt')
#train_images = torch.load('./captures/filtered_raw.pt')
#train_masks = torch.load('./captures/filtered_masked.pt')
#train_boxes = torch.load('./captures/filtered_boxes.pt')

In [ ]:
from mobile_sam import sam_model_registry, SamPredictor

model_type = "vit_t"
sam_checkpoint = "model/MobileSAM_Vortex_checkpoint.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

mobile_sam = sam_model_registry["vit_t"](checkpoint="weights/mobile_sam.pt")
mobile_sam.to(device=device)
predictor = SamPredictor(mobile_sam)

In [ ]:
# Freeze layers in MobileSAM
for name, param in mobile_sam.named_parameters():
    if name.startswith("image_encoder"):  # Assuming "image_encoder" corresponds to the vision encoder
        param.requires_grad = False  # Freeze the vision encoder
    elif name.startswith("prompt_encoder"):  # Assuming "prompt_encoder" exists in MobileSAM
        param.requires_grad = False  # Freeze the prompt encoder

# Verify trainable parameters
trainable_params = sum(p.numel() for p in mobile_sam.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

# Iterate through all parameters and check their trainable status
print("MobileSAM Parameters:")
for name, param in mobile_sam.named_parameters():
        print(f"{name}: {'Trainable' if param.requires_grad else 'Frozen'}")

# Count the total trainable parameters
trainable_params = sum(p.numel() for p in mobile_sam.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters in MobileSAM: {trainable_params}")

**Training Loop**

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Initialize the optimizer for the mask decoder
optimizer = Adam(mobile_sam.mask_decoder.parameters(), lr=1e-3, weight_decay=0)


from torch import nn
from torch.nn import functional as F


class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):

        # comment out if your model contains a sigmoid or equivalent activation layer
        inputs = F.sigmoid(inputs)

        # flatten label and prediction tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.0 * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        BCE = F.binary_cross_entropy(inputs, targets, reduction="mean")
        Dice_BCE = BCE + dice_loss

        return Dice_BCE


# Define the segmentation loss function
# You can choose between DiceFocalLoss, FocalLoss, or DiceCELoss
seg_loss = DiceBCELoss()

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)

In [ ]:
from tqdm import tqdm
from statistics import mean
import torch
import math

# Training loop
num_epochs = 25
device = "cuda" if torch.cuda.is_available() else "cpu"
mobile_sam.to(device)

mobile_sam.train()
for epoch in range(num_epochs):
    train_losses = []
    for batch_group in tqdm(train_batch, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
        #stacking mask/training model/mask retrieve
        mask_batch = torch.stack([sample["ground_truth_mask"].unsqueeze(0).to(device) for sample in batch_group]) # 8 masked images rgb
        outputs = mobile_sam(batched_input = batch_group, multimask_output = False) #requires the whole batch
        predicted_masks = torch.stack([output["masks"].squeeze(1).float() for output in outputs])

        #loss calculation/optimization
        train_loss = seg_loss(predicted_masks, mask_batch)
        train_losses.append(train_loss.item())
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        if (len(train_losses)) % 100 == 0:
            print(f'Batch {len(train_losses)+1}/{len(train_batch)} - Training Loss: {train_loss.item():.2f}')

    mean_train_loss = mean(train_losses)
    train_losses.clear()

    # Validation loss calculation
    mobile_sam.eval()
    val_losses = []
    with torch.no_grad():
        for batch_group in tqdm(val_batch, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
            mask_batch = torch.stack([sample["ground_truth_mask"].unsqueeze(0).to(device) for sample in batch_group])
            outputs = mobile_sam(batched_input=batch_group, multimask_output=False)
            predicted_masks = torch.stack([output["masks"].squeeze(1).float() for output in outputs])
            val_loss = seg_loss(predicted_masks, mask_batch)
            val_losses.append(val_loss.item())
    mean_val_loss = mean(val_losses)
     
    print(f'EPOCH: {epoch + 1}/{num_epochs} - Train Loss: {mean_train_loss:.4f} | Val Loss: {mean_val_loss:.4f}')
    scheduler.step(val_loss)
    mobile_sam.train()


In [ ]:
# Save the state_dict after moving layers
torch.save(mobile_sam.state_dict(), "model/MobileSAM_Vortex_checkpoint.pth")

**Inference**

In [ ]:
from mobile_sam import sam_model_registry, SamPredictor

vortex_model = sam_model_registry["vit_t"](checkpoint="model/MobileSAM_Vortex_checkpoint.pth")
vortex_model.eval()
predictor = SamPredictor(vortex_model)

In [ ]:
import random
test_number = 3

for i in range(test_number):
    idx = random.randint(0,len(test_images)-1)
    
    test_image = test_images[idx].cpu().numpy()
    test_image = np.transpose(test_image, (1,2,0))
    test_box = test_boxes[idx].cpu().numpy()
    test_mask = test_masks[idx].cpu().numpy()

    predictor.set_image(test_image)
    masks, _, _ = predictor.predict(box = test_box, return_logits=True)
    
    masks = np.transpose(masks,(1,2,0))
    masks = masks.astype(np.uint8)
    fig, axes = plt.subplots(1,3,figsize=(15,5))

    axes[0].imshow(test_image)
    axes[0].set_title("Original Image")

    axes[1].imshow(test_mask, cmap='gray')
    axes[1].set_title("Ground Truth Mask")

    axes[2].imshow(masks)
    axes[2].set_title("Inference Output")

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    plt.show()


In [ ]:
torch.cuda.empty_cache()